# 15. Claw-Neighbourhood And C9-ULK Coordination

I use donor-level pseudobulk summaries to address two linked questions:

1. whether the transcriptional neighbourhood of the FIP200 Claw domain is selectively altered in C9orf72-ALS,
2. whether the C9orf72 complex and ULK1/2 initiation machinery are transcriptionally coordinated, and whether that coordination is disrupted in C9orf72-ALS.

I reuse saved checkpoints and saved broad-class outputs where possible, and I keep all interpretation-sensitive choices in explicit placeholder settings blocks.


In [1]:
from pathlib import Path
import itertools
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scipy import sparse
from scipy.stats import zscore
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

MERGED_PB_PATH = PROJECT_ROOT / 'data' / 'processed' / 'merged' / 'adata_merged_pb.h5ad'
BROAD_PB_DIR = PROJECT_ROOT / 'results' / 'pseudobulk' / 'broad'
DE_BROAD_DIR = PROJECT_ROOT / 'results' / 'de_pseudobulk_broad'
WGCNA_BROAD_DIR = PROJECT_ROOT / 'results' / 'wgcna_broad'

RESULTS_15_DIR = PROJECT_ROOT / 'results' / '15_claw_neighbourhood_coordination'
FIG_DIR = RESULTS_15_DIR / 'figures'
TABLE_DIR = RESULTS_15_DIR / 'tables'
INTERMEDIATE_DIR = RESULTS_15_DIR / 'intermediate'

for path in [RESULTS_15_DIR, FIG_DIR, TABLE_DIR, INTERMEDIATE_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## 1. Load Data And Define The Analysis Scope

I keep the input scope parallel to notebook 14: the main full-gene annotated object is `adata_merged_pb.h5ad`, which already contains the broad and refined annotations needed for donor-level pseudobulk summaries.


In [2]:
CLAW_PANEL_INFO = pd.DataFrame([
    {'symbol': 'ATG16L1', 'functional_group': 'ATG8_lipidation_interface', 'alias_comment': ''},
    {'symbol': 'SQSTM1', 'functional_group': 'cargo_receptor', 'alias_comment': 'p62'},
    {'symbol': 'OPTN', 'functional_group': 'cargo_receptor_TBK1_axis', 'alias_comment': 'OPTN'},
    {'symbol': 'CALCOCO2', 'functional_group': 'cargo_receptor', 'alias_comment': 'NDP52'},
    {'symbol': 'TAX1BP1', 'functional_group': 'cargo_receptor_TBK1_axis', 'alias_comment': 'TAX1BP1'},
    {'symbol': 'NBR1', 'functional_group': 'cargo_receptor', 'alias_comment': 'NBR1'},
    {'symbol': 'CCPG1', 'functional_group': 'ER_phagy_receptor', 'alias_comment': 'CCPG1'},
    {'symbol': 'AZI2', 'functional_group': 'TBK1_adaptor', 'alias_comment': 'NAP1 gene symbol'},
    {'symbol': 'TBKBP1', 'functional_group': 'TBK1_adaptor', 'alias_comment': 'SINTBAD gene symbol'},
    {'symbol': 'TNIP1', 'functional_group': 'TBK1_regulatory_context', 'alias_comment': 'TNIP1'},
    {'symbol': 'FAM134B', 'functional_group': 'ER_phagy_receptor', 'alias_comment': 'RETREG1 / FAM134B'},
])

COORDINATION_PANEL_INFO = pd.DataFrame([
    {'symbol': 'C9ORF72', 'functional_group': 'C9_complex', 'alias_comment': ''},
    {'symbol': 'SMCR8', 'functional_group': 'C9_complex', 'alias_comment': ''},
    {'symbol': 'WDR41', 'functional_group': 'C9_complex', 'alias_comment': ''},
    {'symbol': 'RB1CC1', 'functional_group': 'ULK_initiation_complex', 'alias_comment': 'FIP200'},
    {'symbol': 'ATG13', 'functional_group': 'ULK_initiation_complex', 'alias_comment': ''},
    {'symbol': 'ATG101', 'functional_group': 'ULK_initiation_complex', 'alias_comment': ''},
    {'symbol': 'ULK1', 'functional_group': 'ULK_kinase', 'alias_comment': ''},
    {'symbol': 'ULK2', 'functional_group': 'ULK_kinase', 'alias_comment': ''},
])

CLAW_PANEL = CLAW_PANEL_INFO['symbol'].tolist()
COORDINATION_PANEL = COORDINATION_PANEL_INFO['symbol'].tolist()
UNION_PANEL = list(dict.fromkeys(CLAW_PANEL + COORDINATION_PANEL))

adata_master = ad.read_h5ad(MERGED_PB_PATH, backed='r')

required_obs_cols = ['sample', 'condition', 'cell_class_major_harmony', 'cell_subtype']
missing_obs_cols = [col for col in required_obs_cols if col not in adata_master.obs.columns]
if missing_obs_cols:
    raise KeyError(f'Missing required annotation columns in {MERGED_PB_PATH.name}: {missing_obs_cols}')

scope_summary = pd.DataFrame({
    'checkpoint': [MERGED_PB_PATH.name],
    'n_cells': [adata_master.n_obs],
    'n_genes': [adata_master.n_vars],
    'has_counts_layer': ['counts' in adata_master.layers.keys()],
    'n_samples': [adata_master.obs['sample'].nunique()],
    'conditions': [', '.join(map(str, sorted(adata_master.obs['condition'].astype(str).unique())))],
    'broad_classes': [', '.join(map(str, sorted(adata_master.obs['cell_class_major_harmony'].astype(str).dropna().unique())))],
    'n_refined_subtypes': [adata_master.obs['cell_subtype'].dropna().astype(str).nunique()],
    'claw_panel_size': [len(CLAW_PANEL)],
    'coordination_panel_size': [len(COORDINATION_PANEL)],
})

display(scope_summary)
print('Top broad classes:')
display(adata_master.obs['cell_class_major_harmony'].value_counts(dropna=False).to_frame('n_cells'))
print('Top refined subtypes:')
display(adata_master.obs['cell_subtype'].value_counts(dropna=False).head(15).to_frame('n_cells'))


,checkpoint,n_cells,n_genes,has_counts_layer,n_samples,conditions,broad_classes,n_refined_subtypes,claw_panel_size,coordination_panel_size
0,adata_merged_pb.h5ad,88883,61552,True,12,"Control, c9ALS, sALS","Excitatory, Glia, Inhibitory, Vascular",22,11,8


Top broad classes:


,n_cells
cell_class_major_harmony,
Glia,65529
Excitatory,16001
Inhibitory,6717
Vascular,636


Top refined subtypes:


,n_cells
cell_subtype,
Glia.Oligo,50041
Ex.L2_L3.CUX2_RASGRF2,6768
Glia.Astro.GFAP.neg,5246
Glia.OPC,5143
Glia.Micro,3327
In.5HT3aR.DISC1_RELN,2464
Ex.L4_L6.RORB_LRRK1,1908
In.PV.PVALB_MYBPC1,1884
Ex.L5_L6.THEMIS_TMEM233,1818


## 2. Gene Availability And Panel Definition

I first check whether all requested panel genes are available in the full-gene annotated object that I will use for donor-level summarisation.


In [3]:
def build_gene_availability_table(var_names, requested_genes):
    var_names = pd.Index(var_names.astype(str))
    upper_lookup = {gene.upper(): gene for gene in var_names}
    rows = []
    requested_to_var = {}

    for gene in requested_genes:
        if gene in var_names:
            matched_gene = gene
            match_type = 'exact'
            present = True
        elif gene.upper() in upper_lookup:
            matched_gene = upper_lookup[gene.upper()]
            match_type = 'case_insensitive'
            present = True
        else:
            matched_gene = pd.NA
            match_type = 'missing'
            present = False

        if present:
            requested_to_var[gene] = matched_gene

        rows.append({
            'requested_gene': gene,
            'matched_var_name': matched_gene,
            'match_type': match_type,
            'present': present,
        })

    availability_df = pd.DataFrame(rows)
    return availability_df, requested_to_var


def annotate_panel_availability(panel_info_df, availability_df):
    return panel_info_df.merge(
        availability_df.rename(columns={'requested_gene': 'symbol'}),
        on='symbol',
        how='left',
    )


union_availability_df, requested_to_var = build_gene_availability_table(adata_master.var_names, UNION_PANEL)
present_union_genes = union_availability_df.loc[union_availability_df['present'], 'requested_gene'].tolist()
missing_union_genes = union_availability_df.loc[~union_availability_df['present'], 'requested_gene'].tolist()
var_to_requested = {v: k for k, v in requested_to_var.items()}

claw_availability_df = annotate_panel_availability(
    CLAW_PANEL_INFO,
    union_availability_df.loc[union_availability_df['requested_gene'].isin(CLAW_PANEL)].copy(),
)
coordination_availability_df = annotate_panel_availability(
    COORDINATION_PANEL_INFO,
    union_availability_df.loc[union_availability_df['requested_gene'].isin(COORDINATION_PANEL)].copy(),
)

present_claw_genes = claw_availability_df.loc[claw_availability_df['present'], 'symbol'].tolist()
missing_claw_genes = claw_availability_df.loc[~claw_availability_df['present'], 'symbol'].tolist()
present_coordination_genes = coordination_availability_df.loc[coordination_availability_df['present'], 'symbol'].tolist()
missing_coordination_genes = coordination_availability_df.loc[~coordination_availability_df['present'], 'symbol'].tolist()

union_availability_df.to_csv(TABLE_DIR / '15_union_panel_gene_availability.csv', index=False)
claw_availability_df.to_csv(TABLE_DIR / '15_claw_gene_availability.csv', index=False)
coordination_availability_df.to_csv(TABLE_DIR / '15_coordination_gene_availability.csv', index=False)

display(claw_availability_df)
display(coordination_availability_df)

if missing_union_genes:
    print('Missing genes will be excluded from downstream summaries:')
    print(missing_union_genes)
else:
    print('All requested genes are available after exact/case-insensitive matching.')

adata_union = adata_master[:, [requested_to_var[g] for g in present_union_genes]].to_memory()
adata_union.var['requested_gene'] = [var_to_requested[g] for g in adata_union.var_names]


,symbol,functional_group,alias_comment,matched_var_name,match_type,present
0,ATG16L1,ATG8_lipidation_interface,,ATG16L1,exact,True
1,SQSTM1,cargo_receptor,p62,SQSTM1,exact,True
2,OPTN,cargo_receptor_TBK1_axis,OPTN,OPTN,exact,True
3,CALCOCO2,cargo_receptor,NDP52,CALCOCO2,exact,True
4,TAX1BP1,cargo_receptor_TBK1_axis,TAX1BP1,TAX1BP1,exact,True
5,NBR1,cargo_receptor,NBR1,NBR1,exact,True
6,CCPG1,ER_phagy_receptor,CCPG1,CCPG1,exact,True
7,AZI2,TBK1_adaptor,NAP1 gene symbol,AZI2,exact,True
8,TBKBP1,TBK1_adaptor,SINTBAD gene symbol,TBKBP1,exact,True
9,TNIP1,TBK1_regulatory_context,TNIP1,TNIP1,exact,True


,symbol,functional_group,alias_comment,matched_var_name,match_type,present
0,C9ORF72,C9_complex,,C9orf72,case_insensitive,True
1,SMCR8,C9_complex,,SMCR8,exact,True
2,WDR41,C9_complex,,WDR41,exact,True
3,RB1CC1,ULK_initiation_complex,FIP200,RB1CC1,exact,True
4,ATG13,ULK_initiation_complex,,ATG13,exact,True
5,ATG101,ULK_initiation_complex,,ATG101,exact,True
6,ULK1,ULK_kinase,,ULK1,exact,True
7,ULK2,ULK_kinase,,ULK2,exact,True


Missing genes will be excluded from downstream summaries:
['FAM134B']


## 3. Placeholder Settings For Manual Review

I keep all threshold-sensitive and ranking-sensitive choices explicit here so that later interpretation remains transparent.


In [4]:
MIN_CELLS_PER_SUBTYPE_SAMPLE = 20
MIN_CONTROL_DONORS_FOR_SUBTYPE_BASELINE = 3
MIN_DONORS_FOR_CORRELATION = 4
MIN_DONORS_PER_CONDITION_FOR_SUBTYPE_DE = 3

CLAW_PERTURBED_PADJ_THRESHOLD = 0.10
CLAW_PERTURBED_LFC_THRESHOLD = 0.25

TOP_N_SUBTYPES_TO_PLOT = 15
TOP_N_SUBTYPES_TO_PLOT_FOR_DE = 15
TOP_N_SUBTYPES_TO_PLOT_FOR_SCORE = 15

CLAW_PROGRAMME_GENE_WEIGHTS = {gene: 1.0 for gene in present_claw_genes}

FOLLOWUP_SCORE_WEIGHTS = {
    'claw_baseline': 1.0,
    'claw_perturbation': 1.0,
    'coordination_baseline': 1.0,
    'coordination_disruption': 1.0,
    'robustness': 0.5,
}

SELECTED_BROAD_COORDINATION_GROUPS = ['Excitatory', 'Glia']
SELECTED_SUBTYPE_COORDINATION_GROUPS = ['Glia.Micro']
SELECTED_SCATTER_GROUPS_BROAD = ['Excitatory', 'Glia']
SELECTED_SCATTER_GROUPS_SUBTYPE = []
SELECTED_SCATTER_PAIRS = [
    ('SMCR8', 'RB1CC1'),
    ('C9ORF72', 'ATG13'),
    ('SMCR8', 'ATG101'),
    ('WDR41', 'RB1CC1'),
]

OPTIONAL_COMPARATOR_PANELS = {}

DE_COMPARISON_CONFIG = {
    'c9ALS_vs_Control': {'file_stem': 'Control_vs_c9ALS', 'test': 'c9ALS', 'ref': 'Control'},
    'c9ALS_vs_sALS': {'file_stem': 'sALS_vs_c9ALS', 'test': 'c9ALS', 'ref': 'sALS'},
    'sALS_vs_Control': {'file_stem': 'Control_vs_sALS', 'test': 'sALS', 'ref': 'Control'},
}


## 4. Helper Functions

I reuse the donor-level summarisation style from notebook 14, but I generalise the helpers so I can apply them to the Claw-neighbourhood panel and the C9-ULK coordination panel separately.


In [5]:
def matrix_to_dataframe(matrix, obs_names, var_names):
    if sparse.issparse(matrix):
        matrix = matrix.toarray()
    return pd.DataFrame(matrix, index=pd.Index(obs_names, name='obs_name'), columns=list(var_names))


union_counts_df = matrix_to_dataframe(
    adata_union.layers['counts'] if 'counts' in adata_union.layers else adata_union.X,
    adata_union.obs_names,
    adata_union.var['requested_gene'].tolist(),
)
union_obs_df = adata_union.obs[['sample', 'condition', 'cell_class_major_harmony', 'cell_subtype']].copy()
union_obs_df.index = union_counts_df.index


BROAD_CLASS_ORDER = ['Excitatory', 'Inhibitory', 'Glia', 'Vascular']
CONDITION_ORDER = ['Control', 'sALS', 'c9ALS']
C9_COMPLEX_GENES = [gene for gene in ['C9ORF72', 'SMCR8', 'WDR41'] if gene in present_coordination_genes]
ULK_COMPLEX_GENES = [gene for gene in ['RB1CC1', 'ATG13', 'ATG101', 'ULK1', 'ULK2'] if gene in present_coordination_genes]


def sanitize_label(text):
    return str(text).replace('/', '_').replace(' ', '_').replace(':', '_')


def save_figure(fig, stem):
    png_path = FIG_DIR / f'{stem}.png'
    pdf_path = FIG_DIR / f'{stem}.pdf'
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, dpi=300, bbox_inches='tight')
    plt.close(fig)


def counts_to_cpm_and_logcpm(counts_df):
    lib_sizes = counts_df.sum(axis=1)
    lib_sizes = lib_sizes.replace(0, np.nan)
    cpm = counts_df.div(lib_sizes, axis=0) * 1e6
    cpm = cpm.fillna(0.0)
    logcpm = np.log1p(cpm)
    return cpm, logcpm


def compute_weighted_programme_score(mean_expr_df, weights):
    if mean_expr_df.empty:
        return pd.Series(dtype=float)

    present_weights = pd.Series(weights, dtype=float)
    present_weights = present_weights.loc[present_weights.index.intersection(mean_expr_df.columns)]
    if present_weights.empty:
        raise ValueError('No overlap between programme score weights and mean expression columns.')

    expr_use = mean_expr_df.loc[:, present_weights.index]
    z_expr = expr_use.apply(lambda col: pd.Series(zscore(col, nan_policy='omit'), index=col.index), axis=0)
    z_expr = z_expr.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    score = z_expr.mul(present_weights, axis=1).sum(axis=1) / present_weights.sum()
    return score


def compute_programme_score_by_donor(logcpm_df, panel_genes, weights):
    genes = [gene for gene in panel_genes if gene in logcpm_df.columns]
    if not genes:
        return pd.Series(dtype=float)
    expr_use = logcpm_df.loc[:, genes]
    z_expr = expr_use.apply(lambda col: pd.Series(zscore(col, nan_policy='omit'), index=col.index), axis=0)
    z_expr = z_expr.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    weights_use = pd.Series(weights, dtype=float).reindex(genes).fillna(1.0)
    return z_expr.mul(weights_use, axis=1).sum(axis=1) / weights_use.sum()


def save_heatmap(df, title, stem, cmap='mako', center=None, figsize_scale=(0.75, 0.55), xlabel='', ylabel=''):
    if df.empty:
        return
    fig_w = max(8, df.shape[1] * figsize_scale[0] + 4)
    fig_h = max(4, df.shape[0] * figsize_scale[1] + 2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(df, cmap=cmap, center=center, linewidths=0.5, linecolor='white', ax=ax)
    ax.set_title(title)
    ax.set_xlabel(xlabel or (df.columns.name if df.columns.name else ''))
    ax.set_ylabel(ylabel or (df.index.name if df.index.name else ''))
    fig.tight_layout()
    save_figure(fig, stem)


def save_dotplot_summary(mean_expr_df, fraction_df, title, stem):
    if mean_expr_df.empty:
        return

    long_df = (
        mean_expr_df.rename_axis(index='group_name', columns='gene')
        .stack()
        .rename('mean_logCPM')
        .reset_index()
    )
    long_df = long_df.merge(
        fraction_df.rename_axis(index='group_name', columns='gene').stack().rename('fraction_detected').reset_index(),
        on=['group_name', 'gene'],
        how='left',
    )

    x_order = list(mean_expr_df.columns)
    y_order = list(mean_expr_df.index)
    x_pos = {gene: i for i, gene in enumerate(x_order)}
    y_pos = {group: i for i, group in enumerate(y_order)}

    fig, ax = plt.subplots(figsize=(max(8, len(x_order) * 1.0 + 2), max(4, len(y_order) * 0.5 + 2)))
    scatter = ax.scatter(
        long_df['gene'].map(x_pos),
        long_df['group_name'].map(y_pos),
        s=80 + 420 * long_df['fraction_detected'].fillna(0.0),
        c=long_df['mean_logCPM'],
        cmap='mako',
        edgecolor='black',
        linewidth=0.4,
    )
    fig.colorbar(scatter, label='Mean logCPM', ax=ax)
    ax.set_xticks(range(len(x_order)))
    ax.set_xticklabels(x_order, rotation=45, ha='right')
    ax.set_yticks(range(len(y_order)))
    ax.set_yticklabels(y_order)
    ax.set_title(title)
    ax.set_xlabel('Panel gene')
    ax.set_ylabel('Cell type')
    fig.tight_layout()
    save_figure(fig, stem)


def build_ranking_table(mean_expr_df, score_series, n_donors_series, level_name, score_label):
    ranking_rows = []
    for gene in mean_expr_df.columns:
        ranked = mean_expr_df[gene].sort_values(ascending=False)
        for rank, (group_name, value) in enumerate(ranked.items(), start=1):
            ranking_rows.append({
                'level': level_name,
                'ranking_type': 'gene_expression',
                'feature': gene,
                'rank': rank,
                'group_name': group_name,
                'value': value,
                'n_control_donors': n_donors_series.get(group_name, np.nan),
            })

    ranked_score = score_series.sort_values(ascending=False)
    for rank, (group_name, value) in enumerate(ranked_score.items(), start=1):
        ranking_rows.append({
            'level': level_name,
            'ranking_type': 'programme_score',
            'feature': score_label,
            'rank': rank,
            'group_name': group_name,
            'value': value,
            'n_control_donors': n_donors_series.get(group_name, np.nan),
        })

    return pd.DataFrame(ranking_rows)


def load_saved_broad_pseudobulk(requested_to_var, group_names=None):
    counts_by_group = {}
    meta_by_group = {}
    group_names = group_names or BROAD_CLASS_ORDER

    for group_name in group_names:
        counts_path = BROAD_PB_DIR / f'{group_name}_counts.csv'
        meta_path = BROAD_PB_DIR / f'{group_name}_meta.csv'
        if not counts_path.exists() or not meta_path.exists():
            warnings.warn(f'Missing saved broad pseudobulk input for {group_name}; skipping.')
            continue

        counts_df = pd.read_csv(counts_path, index_col=0)
        meta_df = pd.read_csv(meta_path)
        counts_df.index = counts_df.index.astype(str)
        meta_df['sample'] = meta_df['sample'].astype(str)

        unique_ids = pd.Index([f'{group_name}__{sample_id}' for sample_id in counts_df.index], name='pseudobulk_id')
        counts_df.index = unique_ids
        meta_df.index = unique_ids

        selected_cols = {}
        for requested_gene, var_name in requested_to_var.items():
            if var_name in counts_df.columns:
                selected_cols[var_name] = requested_gene

        counts_core = counts_df.loc[:, list(selected_cols.keys())].rename(columns=selected_cols)
        meta_df['group_name'] = group_name

        counts_by_group[group_name] = counts_core
        meta_by_group[group_name] = meta_df

    return counts_by_group, meta_by_group


def build_grouped_pseudobulk_from_cells(counts_df, obs_df, groupby_col, genes, min_cells_per_sample):
    work_obs = obs_df[['sample', 'condition', groupby_col]].copy()
    work_obs = work_obs.dropna(subset=[groupby_col]).copy()
    work_obs[groupby_col] = work_obs[groupby_col].astype(str)

    counts_use = counts_df.loc[work_obs.index, genes].copy()
    counts_use['sample'] = work_obs['sample'].astype(str).values
    counts_use['condition'] = work_obs['condition'].astype(str).values
    counts_use[groupby_col] = work_obs[groupby_col].values

    grouped = counts_use.groupby([groupby_col, 'sample', 'condition'], sort=True)
    pb_counts = grouped[genes].sum()
    n_cells = grouped.size().rename('n_cells')

    pb_meta = n_cells.reset_index()
    pb_meta['pseudobulk_id'] = pb_meta.apply(lambda row: f"{row[groupby_col]}__{row['sample']}__{row['condition']}", axis=1)
    pb_meta = pb_meta.set_index('pseudobulk_id')

    pb_counts = pb_counts.reset_index()
    pb_counts['pseudobulk_id'] = pb_counts.apply(lambda row: f"{row[groupby_col]}__{row['sample']}__{row['condition']}", axis=1)
    pb_counts = pb_counts.set_index('pseudobulk_id')
    pb_counts = pb_counts.loc[:, genes]

    keep_ids = pb_meta.index[pb_meta['n_cells'] >= min_cells_per_sample]
    pb_meta = pb_meta.loc[keep_ids].copy()
    pb_counts = pb_counts.loc[keep_ids].copy()

    return pb_counts.astype(int), pb_meta


def build_retention_summary(meta_df, group_col, level_name):
    summary = (
        meta_df.groupby([group_col, 'condition'])
        .agg(
            n_retained_donors=('sample', 'nunique'),
            min_cells=('n_cells', 'min'),
            median_cells=('n_cells', 'median'),
            max_cells=('n_cells', 'max'),
        )
        .reset_index()
        .rename(columns={group_col: 'group_name'})
    )
    summary['level'] = level_name
    return summary


def summarise_baseline_from_pseudobulk(counts_df, meta_df, group_col, level_name, panel_name, panel_genes, score_weights, min_control_donors=1, top_n_plot=None):
    cpm_df, logcpm_df = counts_to_cpm_and_logcpm(counts_df.loc[:, panel_genes])
    control_mask = meta_df['condition'].astype(str).eq('Control')
    control_meta = meta_df.loc[control_mask].copy()
    control_cpm = cpm_df.loc[control_mask].copy()
    control_logcpm = logcpm_df.loc[control_mask].copy()

    tmp_log = control_logcpm.copy()
    tmp_log[group_col] = control_meta[group_col].astype(str).values
    mean_expr = tmp_log.groupby(group_col).mean()

    tmp_fraction = (control_cpm > 1.0).astype(float)
    tmp_fraction[group_col] = control_meta[group_col].astype(str).values
    fraction_df = tmp_fraction.groupby(group_col).mean()

    control_donor_counts = control_meta.groupby(group_col)['sample'].nunique()
    pathway_score = compute_weighted_programme_score(mean_expr, score_weights)

    baseline_summary = pd.DataFrame({
        'group_name': mean_expr.index,
        'n_control_donors': control_donor_counts.reindex(mean_expr.index).values,
        'baseline_programme_score': pathway_score.reindex(mean_expr.index).values,
        'baseline_mean_logCPM': mean_expr.mean(axis=1).reindex(mean_expr.index).values,
    })
    baseline_summary = baseline_summary.sort_values('baseline_programme_score', ascending=False)

    if min_control_donors > 1:
        keep_groups = baseline_summary.loc[baseline_summary['n_control_donors'] >= min_control_donors, 'group_name'].tolist()
        baseline_summary = baseline_summary.loc[baseline_summary['group_name'].isin(keep_groups)].copy()
        mean_expr = mean_expr.reindex(keep_groups)
        fraction_df = fraction_df.reindex(keep_groups)
        pathway_score = pathway_score.reindex(keep_groups)
        control_donor_counts = control_donor_counts.reindex(keep_groups)

    ranking_table = build_ranking_table(mean_expr, pathway_score, control_donor_counts, level_name, f'{panel_name}_programme_score')

    plot_order = baseline_summary['group_name'].tolist()
    if top_n_plot is not None:
        plot_order = plot_order[:top_n_plot]

    mean_expr_plot = mean_expr.reindex(plot_order)
    fraction_plot = fraction_df.reindex(plot_order)

    mean_expr.to_csv(TABLE_DIR / f'15_{panel_name}_{level_name}_baseline_control_mean_logcpm.csv')
    fraction_df.to_csv(TABLE_DIR / f'15_{panel_name}_{level_name}_baseline_control_fraction_detected.csv')
    baseline_summary.to_csv(TABLE_DIR / f'15_{panel_name}_{level_name}_baseline_control_summary.csv', index=False)
    ranking_table.to_csv(TABLE_DIR / f'15_{panel_name}_{level_name}_baseline_control_rankings.csv', index=False)

    save_heatmap(
        mean_expr_plot,
        title=f'{panel_name.replace("_", " ").title()} | {level_name.replace("_", " ").title()} baseline expression in controls',
        stem=f'15_{panel_name}_{level_name}_baseline_control_heatmap',
        cmap='mako',
    )
    save_dotplot_summary(
        mean_expr_plot,
        fraction_plot,
        title=f'{panel_name.replace("_", " ").title()} | {level_name.replace("_", " ").title()} donor-level baseline summary',
        stem=f'15_{panel_name}_{level_name}_baseline_control_dotplot',
    )

    return {
        'counts': counts_df.loc[:, panel_genes],
        'meta': meta_df.copy(),
        'cpm': cpm_df,
        'logcpm': logcpm_df,
        'mean_expr': mean_expr,
        'fraction': fraction_df,
        'summary': baseline_summary,
        'rankings': ranking_table,
    }


def summarise_programme_scores(logcpm_df, meta_df, group_col, level_name, panel_name, panel_genes, score_weights, top_n_plot=None):
    donor_scores = compute_programme_score_by_donor(logcpm_df.loc[:, panel_genes], panel_genes, score_weights).rename('programme_score')
    donor_df = meta_df[['sample', 'condition', group_col]].copy()
    donor_df['programme_score'] = donor_scores.values
    donor_df = donor_df.rename(columns={group_col: 'group_name'})

    summary_df = (
        donor_df.groupby(['group_name', 'condition'])
        .agg(
            n_donors=('sample', 'nunique'),
            mean_programme_score=('programme_score', 'mean'),
            median_programme_score=('programme_score', 'median'),
            sd_programme_score=('programme_score', 'std'),
        )
        .reset_index()
    )

    donor_df.to_csv(TABLE_DIR / f'15_{panel_name}_{level_name}_programme_scores_by_donor.csv', index=False)
    summary_df.to_csv(TABLE_DIR / f'15_{panel_name}_{level_name}_programme_scores_summary.csv', index=False)

    plot_order = summary_df.groupby('group_name')['mean_programme_score'].mean().sort_values(ascending=False).index.tolist()
    if top_n_plot is not None:
        plot_order = plot_order[:top_n_plot]

    heatmap_df = (
        summary_df.loc[summary_df['group_name'].isin(plot_order)]
        .pivot(index='group_name', columns='condition', values='mean_programme_score')
        .reindex(plot_order)
        .reindex(columns=[c for c in CONDITION_ORDER if c in summary_df['condition'].astype(str).unique()])
    )
    save_heatmap(
        heatmap_df,
        title=f'{panel_name.replace("_", " ").title()} | {level_name.replace("_", " ").title()} programme score by condition',
        stem=f'15_{panel_name}_{level_name}_programme_score_heatmap',
        cmap='vlag',
        center=0,
    )

    if level_name == 'broad':
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.stripplot(
            data=donor_df,
            x='group_name',
            y='programme_score',
            hue='condition',
            order=[g for g in BROAD_CLASS_ORDER if g in donor_df['group_name'].astype(str).unique()],
            hue_order=[c for c in CONDITION_ORDER if c in donor_df['condition'].astype(str).unique()],
            dodge=True,
            size=8,
            ax=ax,
        )
        ax.set_title(f'{panel_name.replace("_", " ").title()} | Broad-class donor programme scores')
        ax.set_xlabel('Broad class')
        ax.set_ylabel('Programme score')
        ax.tick_params(axis='x', rotation=20)
        fig.tight_layout()
        save_figure(fig, f'15_{panel_name}_{level_name}_programme_score_stripplot')

    return donor_df, summary_df, heatmap_df


def load_broad_de_for_panel(panel_name, panel_genes):
    rows = []
    for group_name in BROAD_CLASS_ORDER:
        for comparison_name, cfg in DE_COMPARISON_CONFIG.items():
            fn = DE_BROAD_DIR / f"{group_name}_{cfg['file_stem']}.csv"
            if not fn.exists():
                continue
            df = pd.read_csv(fn)
            if 'gene' not in df.columns:
                continue
            gene_lookup = {gene.upper(): gene for gene in df['gene'].astype(str)}
            for requested_gene in panel_genes:
                matched_gene = gene_lookup.get(requested_gene.upper())
                if matched_gene is None:
                    continue
                hit = df.loc[df['gene'].astype(str).eq(matched_gene)].copy()
                if hit.empty:
                    continue
                hit = hit.iloc[[0]].copy()
                hit['requested_gene'] = requested_gene
                hit['group_name'] = group_name
                hit['comparison'] = comparison_name
                hit['panel_name'] = panel_name
                hit['level'] = 'broad'
                rows.append(hit)

    if not rows:
        return pd.DataFrame(columns=['level', 'group_name', 'comparison', 'requested_gene', 'gene', 'log2FoldChange', 'pvalue', 'padj'])

    out_df = pd.concat(rows, ignore_index=True)
    out_df.to_csv(TABLE_DIR / f'15_{panel_name}_broad_de_long.csv', index=False)
    return out_df


def get_deseq_results_dataframe(stats_obj):
    summary_res = stats_obj.summary()
    if isinstance(summary_res, pd.DataFrame):
        res = summary_res.copy()
    elif hasattr(stats_obj, 'results_df'):
        res = stats_obj.results_df.copy()
    else:
        raise AttributeError('Could not extract DESeq2 results from DeseqStats object.')

    if 'gene' not in res.columns:
        res = res.reset_index().rename(columns={'index': 'gene'})
    return res


def run_panel_deseq_for_one_group(counts_df, meta_df, test_condition, ref_condition):
    keep_mask = meta_df['condition'].astype(str).isin([test_condition, ref_condition])
    counts_use = counts_df.loc[keep_mask].copy()
    meta_use = meta_df.loc[keep_mask, ['condition']].copy()

    if counts_use.empty:
        return pd.DataFrame()

    meta_use['condition'] = pd.Categorical(meta_use['condition'].astype(str), categories=[ref_condition, test_condition])
    counts_use = counts_use.astype(int)

    dds = DeseqDataSet(
        counts=counts_use,
        metadata=meta_use,
        design='~condition',
        refit_cooks=True,
    )
    dds.deseq2()
    stats = DeseqStats(dds, contrast=['condition', test_condition, ref_condition])
    stats.summary()
    return get_deseq_results_dataframe(stats)


def run_subtype_panel_de(counts_df, meta_df, group_col, panel_name, panel_genes, min_donors_per_condition):
    rows = []
    failures = []

    retention = (
        meta_df.groupby([group_col, 'condition'])['sample']
        .nunique()
        .reset_index(name='n_retained_donors')
    )
    retention.to_csv(TABLE_DIR / f'15_{panel_name}_subtype_de_retention_by_condition.csv', index=False)

    for subtype_name in sorted(meta_df[group_col].astype(str).unique()):
        meta_sub = meta_df.loc[meta_df[group_col].astype(str).eq(subtype_name)].copy()
        counts_sub = counts_df.loc[meta_sub.index, panel_genes].copy()

        for comparison_name, cfg in DE_COMPARISON_CONFIG.items():
            donor_counts = meta_sub.groupby('condition')['sample'].nunique().to_dict()
            if donor_counts.get(cfg['test'], 0) < min_donors_per_condition or donor_counts.get(cfg['ref'], 0) < min_donors_per_condition:
                failures.append({
                    'group_name': subtype_name,
                    'comparison': comparison_name,
                    'status': 'insufficient_donors',
                    'n_test_donors': donor_counts.get(cfg['test'], 0),
                    'n_ref_donors': donor_counts.get(cfg['ref'], 0),
                })
                continue

            try:
                res = run_panel_deseq_for_one_group(counts_sub, meta_sub, cfg['test'], cfg['ref'])
                if res.empty:
                    failures.append({
                        'group_name': subtype_name,
                        'comparison': comparison_name,
                        'status': 'empty_result',
                        'n_test_donors': donor_counts.get(cfg['test'], 0),
                        'n_ref_donors': donor_counts.get(cfg['ref'], 0),
                    })
                    continue
                res['requested_gene'] = res['gene'].astype(str).map(lambda g: g if g in panel_genes else var_to_requested.get(g, g))
                res['group_name'] = subtype_name
                res['comparison'] = comparison_name
                res['panel_name'] = panel_name
                res['level'] = 'subtype'
                rows.append(res)
            except Exception as exc:
                failures.append({
                    'group_name': subtype_name,
                    'comparison': comparison_name,
                    'status': f'failed: {exc}',
                    'n_test_donors': donor_counts.get(cfg['test'], 0),
                    'n_ref_donors': donor_counts.get(cfg['ref'], 0),
                })

    de_long_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    failure_df = pd.DataFrame(failures)
    de_long_df.to_csv(TABLE_DIR / f'15_{panel_name}_subtype_de_long.csv', index=False)
    failure_df.to_csv(TABLE_DIR / f'15_{panel_name}_subtype_de_failures.csv', index=False)
    return de_long_df, failure_df


def build_perturbation_summary(de_long_df, panel_name, level_name, padj_threshold, lfc_threshold, top_n_plot=None):
    if de_long_df.empty:
        empty_cols = ['group_name', 'comparison', 'n_panel_genes_tested', 'n_perturbed_genes', 'mean_abs_log2fc', 'mean_abs_log2fc_perturbed', 'level']
        return pd.DataFrame(columns=empty_cols), {}

    work = de_long_df.copy()
    work['perturbed'] = (
        work['padj'].notna()
        & work['log2FoldChange'].notna()
        & (work['padj'] <= padj_threshold)
        & (work['log2FoldChange'].abs() >= lfc_threshold)
    )

    summary = (
        work.groupby(['group_name', 'comparison'])
        .agg(
            n_panel_genes_tested=('requested_gene', 'nunique'),
            n_perturbed_genes=('perturbed', 'sum'),
            mean_abs_log2fc=('log2FoldChange', lambda s: np.nanmean(np.abs(s))),
        )
        .reset_index()
    )

    mean_abs_log2fc_perturbed = (
        work.loc[work['perturbed']]
        .groupby(['group_name', 'comparison'])['log2FoldChange']
        .apply(lambda s: np.nanmean(np.abs(s)))
        .rename('mean_abs_log2fc_perturbed')
        .reset_index()
    )
    summary = summary.merge(mean_abs_log2fc_perturbed, on=['group_name', 'comparison'], how='left')
    summary['level'] = level_name
    summary.to_csv(TABLE_DIR / f'15_{panel_name}_{level_name}_perturbation_summary.csv', index=False)

    heatmaps = {}
    for comparison_name in work['comparison'].dropna().unique():
        heatmap_df = (
            work.loc[work['comparison'].eq(comparison_name), ['group_name', 'requested_gene', 'log2FoldChange']]
            .pivot(index='group_name', columns='requested_gene', values='log2FoldChange')
        )
        group_order = (
            summary.loc[summary['comparison'].eq(comparison_name)]
            .sort_values(['n_perturbed_genes', 'mean_abs_log2fc'], ascending=[False, False])['group_name']
            .tolist()
        )
        if top_n_plot is not None:
            group_order = group_order[:top_n_plot]
        heatmap_plot = heatmap_df.reindex(group_order)
        heatmaps[comparison_name] = heatmap_df
        save_heatmap(
            heatmap_plot,
            title=f'{panel_name.replace("_", " ").title()} | {level_name.title()} {comparison_name} effect sizes',
            stem=f'15_{panel_name}_{level_name}_perturbation_heatmap_{comparison_name}',
            cmap='vlag',
            center=0,
        )

    work.to_csv(TABLE_DIR / f'15_{panel_name}_{level_name}_perturbation_long_table.csv', index=False)
    return summary, heatmaps


def mean_pairwise_correlation(corr_df):
    if corr_df.shape[0] < 2:
        return np.nan
    upper_idx = np.triu_indices_from(corr_df, k=1)
    values = corr_df.to_numpy()[upper_idx]
    if values.size == 0:
        return np.nan
    return float(np.nanmean(values))


def mean_selected_pairwise_correlation(corr_df, genes_a, genes_b=None):
    genes_a = [g for g in genes_a if g in corr_df.index]
    if genes_b is None:
        if len(genes_a) < 2:
            return np.nan
        sub = corr_df.loc[genes_a, genes_a]
        return mean_pairwise_correlation(sub)
    genes_b = [g for g in genes_b if g in corr_df.columns]
    if not genes_a or not genes_b:
        return np.nan
    vals = corr_df.loc[genes_a, genes_b].to_numpy().ravel()
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
        return np.nan
    return float(np.nanmean(vals))


def compute_pc1_variance_explained(expr_df):
    if expr_df.shape[0] < 2 or expr_df.shape[1] < 2:
        return np.nan
    x = expr_df.copy()
    x = x.apply(lambda col: pd.Series(zscore(col, nan_policy='omit'), index=col.index), axis=0)
    x = x.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy()
    if x.shape[0] < 2 or x.shape[1] < 2:
        return np.nan
    _, s, _ = np.linalg.svd(x, full_matrices=False)
    if s.size == 0:
        return np.nan
    var = s ** 2
    total = var.sum()
    if np.isclose(total, 0):
        return np.nan
    return float(var[0] / total)


def summarise_coordination(logcpm_df, meta_df, group_col, panel_name, panel_genes, level_name, condition_filter=None, min_donors=4):
    work_meta = meta_df.copy()
    work_logcpm = logcpm_df.loc[:, panel_genes].copy()
    if condition_filter is not None:
        mask = work_meta['condition'].astype(str).eq(condition_filter)
        work_meta = work_meta.loc[mask].copy()
        work_logcpm = work_logcpm.loc[mask].copy()

    rows = []
    corr_mats = {}

    for group_name in sorted(work_meta[group_col].astype(str).unique()):
        idx = work_meta.index[work_meta[group_col].astype(str).eq(group_name)]
        donor_mat = work_logcpm.loc[idx, panel_genes]
        n_donors = donor_mat.shape[0]

        if n_donors < min_donors:
            rows.append({
                'panel_name': panel_name,
                'level': level_name,
                'condition': condition_filter if condition_filter is not None else 'all_conditions',
                'group_name': group_name,
                'n_donors': n_donors,
                'coordination_mean_pairwise_r': np.nan,
                'coordination_mean_pairwise_r_c9_complex': np.nan,
                'coordination_mean_pairwise_r_ulk_complex': np.nan,
                'coordination_mean_pairwise_r_between_complexes': np.nan,
                'coordination_pc1_variance_explained': np.nan,
                'coordination_flag': 'insufficient_donors',
            })
            continue

        corr_df = donor_mat.corr(method='pearson')
        corr_mats[(group_name, condition_filter if condition_filter is not None else 'all_conditions')] = corr_df
        rows.append({
            'panel_name': panel_name,
            'level': level_name,
            'condition': condition_filter if condition_filter is not None else 'all_conditions',
            'group_name': group_name,
            'n_donors': n_donors,
            'coordination_mean_pairwise_r': mean_pairwise_correlation(corr_df),
            'coordination_mean_pairwise_r_c9_complex': mean_selected_pairwise_correlation(corr_df, C9_COMPLEX_GENES),
            'coordination_mean_pairwise_r_ulk_complex': mean_selected_pairwise_correlation(corr_df, ULK_COMPLEX_GENES),
            'coordination_mean_pairwise_r_between_complexes': mean_selected_pairwise_correlation(corr_df, C9_COMPLEX_GENES, ULK_COMPLEX_GENES),
            'coordination_pc1_variance_explained': compute_pc1_variance_explained(donor_mat),
            'coordination_flag': 'tentative_low_n' if n_donors < 5 else 'ok',
        })

        corr_df.to_csv(INTERMEDIATE_DIR / f'15_{panel_name}_{level_name}_{sanitize_label(group_name)}_{sanitize_label(condition_filter if condition_filter is not None else "all_conditions")}_correlation_matrix.csv')

    summary_df = pd.DataFrame(rows)
    return summary_df, corr_mats


def save_condition_heatmap_grid(corr_mats, group_name, level_name, panel_name, conditions):
    available = [cond for cond in conditions if (group_name, cond) in corr_mats]
    if not available:
        return
    fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 5), squeeze=False)
    for ax, cond in zip(axes[0], available):
        sns.heatmap(
            corr_mats[(group_name, cond)].loc[:, :],
            cmap='vlag',
            center=0,
            linewidths=0.5,
            linecolor='white',
            ax=ax,
        )
        ax.set_title(f'{group_name} | {cond}')
        ax.set_xlabel('')
        ax.set_ylabel('')
    fig.suptitle(f'{panel_name.replace("_", " ").title()} | {level_name.title()} condition-specific coordination', y=1.02)
    fig.tight_layout()
    save_figure(fig, f'15_{panel_name}_{level_name}_{sanitize_label(group_name)}_condition_coordination_grid')


def build_difference_matrix(corr_mats, group_name, test_condition, ref_condition, panel_genes):
    key_test = (group_name, test_condition)
    key_ref = (group_name, ref_condition)
    if key_test not in corr_mats or key_ref not in corr_mats:
        return pd.DataFrame()
    return corr_mats[key_test].loc[panel_genes, panel_genes] - corr_mats[key_ref].loc[panel_genes, panel_genes]


def save_difference_heatmap(diff_df, title, stem):
    if diff_df.empty:
        return
    save_heatmap(diff_df, title=title, stem=stem, cmap='vlag', center=0, figsize_scale=(0.9, 0.8))


def plot_selected_gene_pair_scatter(logcpm_df, meta_df, group_col, group_name, gene_x, gene_y, level_name, panel_name):
    if gene_x not in logcpm_df.columns or gene_y not in logcpm_df.columns:
        return
    idx = meta_df.index[meta_df[group_col].astype(str).eq(group_name)]
    if len(idx) == 0:
        return
    plot_df = meta_df.loc[idx, ['sample', 'condition']].copy()
    plot_df[gene_x] = logcpm_df.loc[idx, gene_x].values
    plot_df[gene_y] = logcpm_df.loc[idx, gene_y].values

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(
        data=plot_df,
        x=gene_x,
        y=gene_y,
        hue='condition',
        hue_order=[c for c in CONDITION_ORDER if c in plot_df['condition'].astype(str).unique()],
        s=90,
        ax=ax,
    )
    ax.set_title(f'{group_name} | {gene_x} vs {gene_y}')
    fig.tight_layout()
    save_figure(fig, f'15_{panel_name}_{level_name}_{sanitize_label(group_name)}_{gene_x}_vs_{gene_y}_scatter')


def minmax_scale(series):
    s = series.astype(float).copy()
    if s.notna().sum() == 0:
        return pd.Series(0.0, index=s.index)
    min_val = s.min(skipna=True)
    max_val = s.max(skipna=True)
    if pd.isna(min_val) or pd.isna(max_val) or np.isclose(max_val, min_val):
        return pd.Series(0.5, index=s.index)
    return (s - min_val) / (max_val - min_val)


def build_followup_summary(level_name, claw_baseline_summary, claw_perturbation_summary, coord_control_summary, coord_diff_summary, retention_summary):
    out = claw_baseline_summary[['group_name', 'baseline_programme_score', 'n_control_donors']].copy()

    if claw_perturbation_summary.empty:
        claw_perturbation_summary = pd.DataFrame(columns=['group_name', 'comparison', 'mean_abs_log2fc', 'n_perturbed_genes'])
    if coord_diff_summary.empty:
        coord_diff_summary = pd.DataFrame(columns=['group_name', 'coordination_delta_mean_pairwise_r', 'coordination_delta_pc1_variance', 'coordination_disruption_abs_delta'])
    out = out.rename(columns={'baseline_programme_score': 'claw_baseline_programme_score'})

    perturbation_focus = (
        claw_perturbation_summary.loc[claw_perturbation_summary['comparison'].isin(['c9ALS_vs_Control', 'c9ALS_vs_sALS'])]
        .groupby('group_name')
        .agg(
            claw_perturbation_metric=('mean_abs_log2fc', 'mean'),
            claw_n_perturbed_genes=('n_perturbed_genes', 'sum'),
        )
        .reset_index()
    )
    out = out.merge(perturbation_focus, on='group_name', how='left')

    coord_control_focus = coord_control_summary[['group_name', 'coordination_mean_pairwise_r', 'coordination_pc1_variance_explained']].copy()
    coord_control_focus = coord_control_focus.rename(columns={
        'coordination_mean_pairwise_r': 'coordination_baseline_mean_pairwise_r',
        'coordination_pc1_variance_explained': 'coordination_baseline_pc1_variance_explained',
    })
    out = out.merge(coord_control_focus, on='group_name', how='left')
    out = out.merge(coord_diff_summary, on='group_name', how='left')

    robustness = retention_summary.groupby('group_name')['n_retained_donors'].min().rename('min_retained_donors').reset_index()
    out = out.merge(robustness, on='group_name', how='left')

    out['claw_baseline_scaled'] = minmax_scale(out['claw_baseline_programme_score'])
    out['claw_perturbation_scaled'] = minmax_scale(out['claw_perturbation_metric'].fillna(0.0))
    out['coordination_baseline_scaled'] = minmax_scale(out['coordination_baseline_mean_pairwise_r'].fillna(0.0))
    out['coordination_disruption_scaled'] = minmax_scale(out['coordination_disruption_abs_delta'].fillna(0.0))
    out['robustness_scaled'] = minmax_scale(out['min_retained_donors'].fillna(0.0))

    out['followup_score'] = (
        FOLLOWUP_SCORE_WEIGHTS['claw_baseline'] * out['claw_baseline_scaled']
        + FOLLOWUP_SCORE_WEIGHTS['claw_perturbation'] * out['claw_perturbation_scaled']
        + FOLLOWUP_SCORE_WEIGHTS['coordination_baseline'] * out['coordination_baseline_scaled']
        + FOLLOWUP_SCORE_WEIGHTS['coordination_disruption'] * out['coordination_disruption_scaled']
        + FOLLOWUP_SCORE_WEIGHTS['robustness'] * out['robustness_scaled']
    )
    out['level'] = level_name
    out = out.sort_values('followup_score', ascending=False).reset_index(drop=True)
    out['followup_rank'] = np.arange(1, len(out) + 1)
    out.to_csv(TABLE_DIR / f'15_{level_name}_followup_prioritisation_summary.csv', index=False)
    return out


def extract_coordination_difference_summary(condition_coordination_df, comparison_pairs):
    rows = []
    if condition_coordination_df.empty:
        return pd.DataFrame(columns=['group_name', 'comparison', 'coordination_delta_mean_pairwise_r', 'coordination_delta_pc1_variance', 'coordination_disruption_abs_delta'])

    for group_name in sorted(condition_coordination_df['group_name'].astype(str).unique()):
        sub = condition_coordination_df.loc[condition_coordination_df['group_name'].astype(str).eq(group_name)].copy()
        for label, (test_cond, ref_cond) in comparison_pairs.items():
            left = sub.loc[sub['condition'].eq(test_cond)]
            right = sub.loc[sub['condition'].eq(ref_cond)]
            if left.empty or right.empty:
                continue
            rows.append({
                'group_name': group_name,
                'comparison': label,
                'coordination_delta_mean_pairwise_r': left['coordination_mean_pairwise_r'].iloc[0] - right['coordination_mean_pairwise_r'].iloc[0],
                'coordination_delta_pc1_variance': left['coordination_pc1_variance_explained'].iloc[0] - right['coordination_pc1_variance_explained'].iloc[0],
                'coordination_disruption_abs_delta': abs(left['coordination_mean_pairwise_r'].iloc[0] - right['coordination_mean_pairwise_r'].iloc[0]),
            })
    return pd.DataFrame(rows)


def try_read_wgcna_membership_for_panel(group_name, panel_genes):
    group_dir = WGCNA_BROAD_DIR / group_name
    if not group_dir.exists():
        return pd.DataFrame(), f'{group_name}: no_saved_wgcna_directory'

    candidate_files = []
    candidate_files.extend(sorted(group_dir.glob('*module*.csv')))
    candidate_files.extend(sorted(group_dir.glob('*gene*.csv')))
    candidate_files.extend(sorted(group_dir.glob('*membership*.csv')))
    candidate_files = list(dict.fromkeys(candidate_files))
    if not candidate_files:
        return pd.DataFrame(), f'{group_name}: no_plaintext_module_exports_found'

    for fn in candidate_files:
        try:
            df = pd.read_csv(fn)
        except Exception:
            continue
        cols_upper = {col.upper(): col for col in df.columns}
        gene_col = None
        module_col = None
        for candidate in ['GENE', 'GENES', 'GENENAME', 'SYMBOL']:
            if candidate in cols_upper:
                gene_col = cols_upper[candidate]
                break
        for candidate in ['MODULE', 'MODULECOLOR', 'MODULE_COLOR', 'COLOR']:
            if candidate in cols_upper:
                module_col = cols_upper[candidate]
                break
        if gene_col is None or module_col is None:
            continue
        subset = df.loc[df[gene_col].astype(str).str.upper().isin([g.upper() for g in panel_genes]), [gene_col, module_col]].copy()
        if subset.empty:
            continue
        subset = subset.rename(columns={gene_col: 'gene', module_col: 'module'})
        subset['group_name'] = group_name
        subset['source_file'] = fn.name
        return subset, f'{group_name}: matched_plaintext_module_export'

    return pd.DataFrame(), f'{group_name}: no_readable_module_membership_table_found'


## 5. Prepare Donor-Level Inputs

I reuse the saved broad-class pseudobulk counts and metadata where available, and I build refined subtype donor-level pseudobulk only for the union of genes needed in this notebook.


In [6]:
broad_counts_by_group, broad_meta_by_group = load_saved_broad_pseudobulk(requested_to_var)

broad_union_counts = pd.concat(broad_counts_by_group.values(), axis=0)
broad_union_meta = pd.concat(broad_meta_by_group.values(), axis=0)
broad_union_meta.index = broad_union_counts.index
broad_union_meta['group_name'] = broad_union_meta['group_name'].astype(str)

subtype_union_counts, subtype_union_meta = build_grouped_pseudobulk_from_cells(
    union_counts_df,
    union_obs_df,
    groupby_col='cell_subtype',
    genes=present_union_genes,
    min_cells_per_sample=MIN_CELLS_PER_SUBTYPE_SAMPLE,
)

broad_retention_summary = build_retention_summary(broad_union_meta, 'group_name', 'broad')
subtype_retention_summary = build_retention_summary(subtype_union_meta, 'cell_subtype', 'subtype')

broad_retention_summary.to_csv(TABLE_DIR / '15_broad_donor_retention_summary.csv', index=False)
subtype_retention_summary.to_csv(TABLE_DIR / '15_subtype_donor_retention_summary.csv', index=False)

print('Broad donor retention:')
display(broad_retention_summary)
print('Subtype donor retention (top 20 by retained donors):')
display(subtype_retention_summary.sort_values(['n_retained_donors', 'median_cells'], ascending=[False, False]).head(20))


Broad donor retention:


,group_name,condition,n_retained_donors,min_cells,median_cells,max_cells,level
0,Excitatory,Control,4,530,752.5,847,broad
1,Excitatory,c9ALS,4,1346,2333.5,2844,broad
2,Excitatory,sALS,4,152,1009.0,2092,broad
3,Glia,Control,4,2303,4792.5,5228,broad
4,Glia,c9ALS,4,4309,4901.5,5918,broad
5,Glia,sALS,4,4883,6662.5,10175,broad
6,Inhibitory,Control,4,125,253.0,409,broad
7,Inhibitory,c9ALS,4,375,854.0,1410,broad
8,Inhibitory,sALS,4,70,523.5,1067,broad
9,Vascular,Control,3,21,30.0,64,broad


Subtype donor retention (top 20 by retained donors):


,group_name,condition,n_retained_donors,min_cells,median_cells,max_cells,level
46,Glia.Oligo,sALS,4,3508,5494.5,8706,subtype
45,Glia.Oligo,c9ALS,4,3112,3507.5,4585,subtype
44,Glia.Oligo,Control,4,1473,3444.5,3764,subtype
1,Ex.L2_L3.CUX2_RASGRF2,c9ALS,4,540,958.5,1307,subtype
2,Ex.L2_L3.CUX2_RASGRF2,sALS,4,22,507.5,1260,subtype
43,Glia.OPC,sALS,4,252,503.5,569,subtype
31,Glia.Astro.GFAP.neg,c9ALS,4,342,453.0,514,subtype
30,Glia.Astro.GFAP.neg,Control,4,141,432.0,626,subtype
32,Glia.Astro.GFAP.neg,sALS,4,93,426.0,908,subtype
42,Glia.OPC,c9ALS,4,324,382.5,496,subtype


## SECTION A. FIP200 Claw-Neighbourhood

### A2. Baseline Expression Landscape In Controls

I summarise the Claw-neighbourhood panel at the donor level across broad classes and refined subtypes, restricting the baseline comparison to control donors.


In [7]:
claw_broad_baseline = summarise_baseline_from_pseudobulk(
    broad_union_counts,
    broad_union_meta,
    group_col='group_name',
    level_name='broad',
    panel_name='claw_neighbourhood',
    panel_genes=present_claw_genes,
    score_weights=CLAW_PROGRAMME_GENE_WEIGHTS,
    min_control_donors=1,
)

claw_subtype_baseline = summarise_baseline_from_pseudobulk(
    subtype_union_counts,
    subtype_union_meta,
    group_col='cell_subtype',
    level_name='subtype',
    panel_name='claw_neighbourhood',
    panel_genes=present_claw_genes,
    score_weights=CLAW_PROGRAMME_GENE_WEIGHTS,
    min_control_donors=MIN_CONTROL_DONORS_FOR_SUBTYPE_BASELINE,
    top_n_plot=TOP_N_SUBTYPES_TO_PLOT,
)

print('Broad baseline summary:')
display(claw_broad_baseline['summary'])
print('Top subtype baseline summary:')
display(claw_subtype_baseline['summary'].head(20))


Broad baseline summary:


,group_name,n_control_donors,baseline_programme_score,baseline_mean_logCPM
2,Inhibitory,4,0.337912,11.270760
0,Excitatory,4,0.055820,11.208459
1,Glia,4,-0.085769,11.151783
3,Vascular,3,-0.307963,11.008904


Top subtype baseline summary:


,group_name,n_control_donors,baseline_programme_score,baseline_mean_logCPM
14,Glia.OPC,4,0.176532,11.148907
11,Glia.Astro.GFAP.pos,4,0.170542,11.179310
4,Ex.L5.PCP4_NXPH2,3,0.132806,11.253461
8,Ex.L6.TLE4_CCBE1,4,0.115544,11.194287
9,Ex.L6.TLE4_MEGF11,4,0.103183,11.226005
10,Glia.Astro.GFAP.neg,4,0.078055,11.153569
17,In.PV.PVALB_MYBPC1,4,0.077523,11.235977
0,Ex.L2_L3.CUX2_RASGRF2,4,0.055938,11.172023
3,Ex.L4_L6.RORB_LRRK1,4,0.044166,11.229794
18,In.Rosehip.LAMP5_PMEPA1,3,0.024856,11.260456


### A3. Differential Expression And A4. Programme-Level Scoring

I reuse saved broad-class DE results for the Claw-neighbourhood panel, and I compute subtype-level panel DE only where donor retention is adequate. I also summarise a donor-level programme score so I can compare the neighbourhood signal across conditions.


In [ ]:
claw_broad_de = load_broad_de_for_panel('claw_neighbourhood', present_claw_genes)
claw_broad_perturbation_summary, claw_broad_perturbation_heatmaps = build_perturbation_summary(
    claw_broad_de,
    panel_name='claw_neighbourhood',
    level_name='broad',
    padj_threshold=CLAW_PERTURBED_PADJ_THRESHOLD,
    lfc_threshold=CLAW_PERTURBED_LFC_THRESHOLD,
)

claw_subtype_de, claw_subtype_de_failures = run_subtype_panel_de(
    subtype_union_counts,
    subtype_union_meta,
    group_col='cell_subtype',
    panel_name='claw_neighbourhood',
    panel_genes=present_claw_genes,
    min_donors_per_condition=MIN_DONORS_PER_CONDITION_FOR_SUBTYPE_DE,
)
claw_subtype_perturbation_summary, claw_subtype_perturbation_heatmaps = build_perturbation_summary(
    claw_subtype_de,
    panel_name='claw_neighbourhood',
    level_name='subtype',
    padj_threshold=CLAW_PERTURBED_PADJ_THRESHOLD,
    lfc_threshold=CLAW_PERTURBED_LFC_THRESHOLD,
    top_n_plot=TOP_N_SUBTYPES_TO_PLOT_FOR_DE,
)

claw_broad_programme_scores, claw_broad_programme_summary, claw_broad_programme_heatmap = summarise_programme_scores(
    claw_broad_baseline['logcpm'],
    broad_union_meta,
    group_col='group_name',
    level_name='broad',
    panel_name='claw_neighbourhood',
    panel_genes=present_claw_genes,
    score_weights=CLAW_PROGRAMME_GENE_WEIGHTS,
)
claw_subtype_programme_scores, claw_subtype_programme_summary, claw_subtype_programme_heatmap = summarise_programme_scores(
    claw_subtype_baseline['logcpm'],
    subtype_union_meta,
    group_col='cell_subtype',
    level_name='subtype',
    panel_name='claw_neighbourhood',
    panel_genes=present_claw_genes,
    score_weights=CLAW_PROGRAMME_GENE_WEIGHTS,
    top_n_plot=TOP_N_SUBTYPES_TO_PLOT_FOR_SCORE,
)

print('Broad Claw-neighbourhood DE summary:')
display(claw_broad_perturbation_summary)
print('Subtype Claw-neighbourhood DE failures / skipped comparisons:')
display(claw_subtype_de_failures.head(20))
print('Subtype Claw-neighbourhood DE summary:')
display(claw_subtype_perturbation_summary.head(20))


Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.02 seconds.

Fitting dispersion trend curve...
... done in 0.02 seconds.

Fitting MAP dispersions...
... done in 0.01 seconds.

Fitting LFCs...
... done in 0.01 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.01 seconds.

Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.01 seconds.

Fitting dispersion trend curve...
/opt/anaconda3/envs/c9_multiomics/lib/python3.11/site-packages/pydeseq2/dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.01 seconds.

Fitting MAP dispersions...
... done in 0.01 seconds.

Fitting LFCs...
... done in 0.01 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: condition c9ALS vs Control
             baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
ATG16L1    243.348008       -0.042545  0.154125 -0.276045  0.782514  0.818014
SQSTM1     407.391125        0.167806  0.168002  0.998833  0.317876  0.818014
OPTN       602.638173        0.391733  0.148525  2.637478  0.008352  0.083525
CALCOCO2   365.146885       -0.138690  0.174700 -0.793873  0.427269  0.818014
TAX1BP1   1269.534447        0.047427  0.105959  0.447598  0.654444  0.818014
NBR1       238.177207        0.148837  0.167103  0.890689  0.373096  0.818014
CCPG1      106.550720       -0.062317  0.270826 -0.230100  0.818014  0.818014
AZI2       246.693594        0.051551  0.207440  0.248513  0.803738  0.818014
TBKBP1      57.941295       -0.590024  0.366454 -1.610093  0.107378  0.536888
TNIP1      115.352403       -0.153342  0.230626 -0.664894  0.506118  0.818014
Log2 fold change & Wald test p-value: condition c9ALS vs Control
            

Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.01 seconds.

Fitting dispersion trend curve...
... done in 0.01 seconds.

Fitting MAP dispersions...
... done in 0.01 seconds.

Fitting LFCs...
... done in 0.01 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.01 seconds.

Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.01 seconds.

Fitting dispersion trend curve...
/opt/anaconda3/envs/c9_multiomics/lib/python3.11/site-packages/pydeseq2/dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.00 seconds.

Fitting MAP dispersions...
... done in 0.01 seconds.

Fitting LFCs...
... done in 0.01 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: condition sALS vs Control
            baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
ATG16L1   161.686674        0.070862  0.120166  0.589699  0.555392  0.694240
SQSTM1    227.610807       -0.282279  0.115007 -2.454452  0.014110  0.035275
OPTN      425.695463        0.587784  0.172078  3.415807  0.000636  0.006359
CALCOCO2  277.002590        0.224392  0.234807  0.955642  0.339253  0.484647
TAX1BP1   870.464426        0.186854  0.114223  1.635873  0.101866  0.203732
NBR1      152.460231        0.149049  0.133258  1.118503  0.263352  0.438920
CCPG1      74.513510        0.032178  0.189311  0.169973  0.865031  0.865031
AZI2      150.884516       -0.062901  0.211912 -0.296828  0.766598  0.851775
TBKBP1     33.568466       -0.972901  0.323769 -3.004919  0.002657  0.013283
TNIP1      69.994223       -0.445112  0.171804 -2.590812  0.009575  0.031917
Log2 fold change & Wald test p-value: condition sALS vs Control
            baseMean  log

...

,group_name,comparison,n_panel_genes_tested,n_perturbed_genes,mean_abs_log2fc,mean_abs_log2fc_perturbed,level
0,Excitatory,c9ALS_vs_Control,10,0,0.184979,NaN,broad
1,Excitatory,c9ALS_vs_sALS,10,0,0.162766,NaN,broad
2,Excitatory,sALS_vs_Control,10,1,0.265243,1.176340,broad
3,Glia,c9ALS_vs_Control,10,0,0.205502,NaN,broad
4,Glia,c9ALS_vs_sALS,10,0,0.240191,NaN,broad
5,Glia,sALS_vs_Control,10,2,0.363683,0.775243,broad
6,Inhibitory,c9ALS_vs_Control,10,0,0.210518,NaN,broad
7,Inhibitory,c9ALS_vs_sALS,10,0,0.148547,NaN,broad
8,Inhibitory,sALS_vs_Control,10,1,0.243755,0.891465,broad
9,Vascular,c9ALS_vs_Control,10,0,0.388188,NaN,broad


Subtype Claw-neighbourhood DE failures / skipped comparisons:


,group_name,comparison,status,n_test_donors,n_ref_donors
0,Ex.L5.PCP4_NXPH2,c9ALS_vs_sALS,insufficient_donors,4,2
1,Ex.L5.PCP4_NXPH2,sALS_vs_Control,insufficient_donors,2,3
2,Ex.L5.VAT1L_THSD4,c9ALS_vs_Control,insufficient_donors,2,1
3,Ex.L5.VAT1L_THSD4,c9ALS_vs_sALS,insufficient_donors,2,1
4,Ex.L5.VAT1L_THSD4,sALS_vs_Control,insufficient_donors,1,1
5,Ex.L5_L6.THEMIS_NR4A2,c9ALS_vs_sALS,insufficient_donors,4,2
6,Ex.L5_L6.THEMIS_NR4A2,sALS_vs_Control,insufficient_donors,2,3
7,Glia.Astro_Micro_mixed,c9ALS_vs_Control,insufficient_donors,2,2
8,Glia.Astro_Micro_mixed,c9ALS_vs_sALS,insufficient_donors,2,0
9,Glia.Astro_Micro_mixed,sALS_vs_Control,insufficient_donors,0,2


Subtype Claw-neighbourhood DE summary:


,group_name,comparison,n_panel_genes_tested,n_perturbed_genes,mean_abs_log2fc,mean_abs_log2fc_perturbed,level
0,Ex.L2_L3.CUX2_RASGRF2,c9ALS_vs_Control,10,1,0.179427,0.391733,subtype
1,Ex.L2_L3.CUX2_RASGRF2,c9ALS_vs_sALS,10,0,0.217645,NaN,subtype
2,Ex.L2_L3.CUX2_RASGRF2,sALS_vs_Control,10,4,0.301431,0.572019,subtype
3,Ex.L3_L5.CUX2_RORB,c9ALS_vs_Control,10,0,0.270469,NaN,subtype
4,Ex.L3_L5.CUX2_RORB,c9ALS_vs_sALS,10,0,0.298779,NaN,subtype
5,Ex.L3_L5.CUX2_RORB,sALS_vs_Control,10,3,0.406062,0.871734,subtype
6,Ex.L4_L5.RORB_POU3F2,c9ALS_vs_Control,10,0,0.190958,NaN,subtype
7,Ex.L4_L5.RORB_POU3F2,c9ALS_vs_sALS,10,2,0.268974,0.837273,subtype
8,Ex.L4_L5.RORB_POU3F2,sALS_vs_Control,10,3,0.382088,0.916324,subtype
9,Ex.L4_L6.RORB_LRRK1,c9ALS_vs_Control,10,0,0.194518,NaN,subtype


### A5. Optional Comparator-Panel Placeholder

This section is intentionally lightweight. If I later want to compare the Claw-neighbourhood signal against other curated receptor/adaptor panels, I can add them through the `OPTIONAL_COMPARATOR_PANELS` settings block without changing the main notebook structure.


In [9]:
if not OPTIONAL_COMPARATOR_PANELS:
    print('No optional comparator panels are currently defined. I will skip comparator analysis for this run.')
else:
    comparator_preview = pd.DataFrame({
        'panel_name': list(OPTIONAL_COMPARATOR_PANELS.keys()),
        'n_genes': [len(v) for v in OPTIONAL_COMPARATOR_PANELS.values()],
        'genes': [', '.join(v) for v in OPTIONAL_COMPARATOR_PANELS.values()],
    })
    comparator_preview.to_csv(TABLE_DIR / '15_optional_comparator_panel_registry.csv', index=False)
    display(comparator_preview)


No optional comparator panels are currently defined. I will skip comparator analysis for this run.


## SECTION B. Coordination Between The C9 Complex And ULK1/2 Initiation Machinery

### B1. Baseline Coordination In Controls
### B2. Condition-Specific Coordination

I next focus on the C9-ULK coordination panel, asking where the genes move together in controls and whether that correlation structure changes in C9-ALS.


In [10]:
coord_broad_baseline = summarise_baseline_from_pseudobulk(
    broad_union_counts,
    broad_union_meta,
    group_col='group_name',
    level_name='broad',
    panel_name='coordination_panel',
    panel_genes=present_coordination_genes,
    score_weights={gene: 1.0 for gene in present_coordination_genes},
    min_control_donors=1,
)
coord_subtype_baseline = summarise_baseline_from_pseudobulk(
    subtype_union_counts,
    subtype_union_meta,
    group_col='cell_subtype',
    level_name='subtype',
    panel_name='coordination_panel',
    panel_genes=present_coordination_genes,
    score_weights={gene: 1.0 for gene in present_coordination_genes},
    min_control_donors=MIN_CONTROL_DONORS_FOR_SUBTYPE_BASELINE,
    top_n_plot=TOP_N_SUBTYPES_TO_PLOT,
)

coord_broad_control_summary, coord_broad_control_mats = summarise_coordination(
    coord_broad_baseline['logcpm'],
    broad_union_meta,
    group_col='group_name',
    panel_name='coordination_panel',
    panel_genes=present_coordination_genes,
    level_name='broad',
    condition_filter='Control',
    min_donors=MIN_DONORS_FOR_CORRELATION,
)
coord_subtype_control_summary, coord_subtype_control_mats = summarise_coordination(
    coord_subtype_baseline['logcpm'],
    subtype_union_meta,
    group_col='cell_subtype',
    panel_name='coordination_panel',
    panel_genes=present_coordination_genes,
    level_name='subtype',
    condition_filter='Control',
    min_donors=MIN_DONORS_FOR_CORRELATION,
)

coord_broad_control_summary.to_csv(TABLE_DIR / '15_coordination_panel_broad_control_coordination_summary.csv', index=False)
coord_subtype_control_summary.to_csv(TABLE_DIR / '15_coordination_panel_subtype_control_coordination_summary.csv', index=False)

coord_broad_condition_rows = []
coord_broad_condition_mats = {}
for cond in [c for c in CONDITION_ORDER if c in broad_union_meta['condition'].astype(str).unique()]:
    summary_df, corr_mats = summarise_coordination(
        coord_broad_baseline['logcpm'],
        broad_union_meta,
        group_col='group_name',
        panel_name='coordination_panel',
        panel_genes=present_coordination_genes,
        level_name='broad',
        condition_filter=cond,
        min_donors=MIN_DONORS_FOR_CORRELATION,
    )
    coord_broad_condition_rows.append(summary_df)
    coord_broad_condition_mats.update(corr_mats)
coord_broad_condition_summary = pd.concat(coord_broad_condition_rows, ignore_index=True) if coord_broad_condition_rows else pd.DataFrame()
coord_broad_condition_summary.to_csv(TABLE_DIR / '15_coordination_panel_broad_condition_specific_summary.csv', index=False)

coord_subtype_condition_rows = []
coord_subtype_condition_mats = {}
for cond in [c for c in CONDITION_ORDER if c in subtype_union_meta['condition'].astype(str).unique()]:
    summary_df, corr_mats = summarise_coordination(
        coord_subtype_baseline['logcpm'],
        subtype_union_meta,
        group_col='cell_subtype',
        panel_name='coordination_panel',
        panel_genes=present_coordination_genes,
        level_name='subtype',
        condition_filter=cond,
        min_donors=MIN_DONORS_FOR_CORRELATION,
    )
    coord_subtype_condition_rows.append(summary_df)
    coord_subtype_condition_mats.update(corr_mats)
coord_subtype_condition_summary = pd.concat(coord_subtype_condition_rows, ignore_index=True) if coord_subtype_condition_rows else pd.DataFrame()
coord_subtype_condition_summary.to_csv(TABLE_DIR / '15_coordination_panel_subtype_condition_specific_summary.csv', index=False)

for group_name in SELECTED_BROAD_COORDINATION_GROUPS:
    save_condition_heatmap_grid(
        coord_broad_condition_mats,
        group_name=group_name,
        level_name='broad',
        panel_name='coordination_panel',
        conditions=[c for c in CONDITION_ORDER if c in broad_union_meta['condition'].astype(str).unique()],
    )
    diff_control = build_difference_matrix(coord_broad_condition_mats, group_name, 'c9ALS', 'Control', present_coordination_genes)
    diff_sals = build_difference_matrix(coord_broad_condition_mats, group_name, 'c9ALS', 'sALS', present_coordination_genes)
    diff_control.to_csv(INTERMEDIATE_DIR / f'15_coordination_panel_broad_{sanitize_label(group_name)}_c9ALS_minus_Control.csv')
    diff_sals.to_csv(INTERMEDIATE_DIR / f'15_coordination_panel_broad_{sanitize_label(group_name)}_c9ALS_minus_sALS.csv')
    save_difference_heatmap(
        diff_control,
        title=f'{group_name} | C9-ALS minus Control coordination difference',
        stem=f'15_coordination_panel_broad_{sanitize_label(group_name)}_c9ALS_minus_Control_heatmap',
    )
    save_difference_heatmap(
        diff_sals,
        title=f'{group_name} | C9-ALS minus sALS coordination difference',
        stem=f'15_coordination_panel_broad_{sanitize_label(group_name)}_c9ALS_minus_sALS_heatmap',
    )

for group_name in SELECTED_SUBTYPE_COORDINATION_GROUPS:
    save_condition_heatmap_grid(
        coord_subtype_condition_mats,
        group_name=group_name,
        level_name='subtype',
        panel_name='coordination_panel',
        conditions=[c for c in CONDITION_ORDER if c in subtype_union_meta['condition'].astype(str).unique()],
    )
    diff_control = build_difference_matrix(coord_subtype_condition_mats, group_name, 'c9ALS', 'Control', present_coordination_genes)
    diff_sals = build_difference_matrix(coord_subtype_condition_mats, group_name, 'c9ALS', 'sALS', present_coordination_genes)
    diff_control.to_csv(INTERMEDIATE_DIR / f'15_coordination_panel_subtype_{sanitize_label(group_name)}_c9ALS_minus_Control.csv')
    diff_sals.to_csv(INTERMEDIATE_DIR / f'15_coordination_panel_subtype_{sanitize_label(group_name)}_c9ALS_minus_sALS.csv')
    save_difference_heatmap(
        diff_control,
        title=f'{group_name} | C9-ALS minus Control coordination difference',
        stem=f'15_coordination_panel_subtype_{sanitize_label(group_name)}_c9ALS_minus_Control_heatmap',
    )
    save_difference_heatmap(
        diff_sals,
        title=f'{group_name} | C9-ALS minus sALS coordination difference',
        stem=f'15_coordination_panel_subtype_{sanitize_label(group_name)}_c9ALS_minus_sALS_heatmap',
    )

print('Broad control-only coordination summary:')
display(coord_broad_control_summary)
print('Subtype control-only coordination summary:')
display(coord_subtype_control_summary.head(20))


Broad control-only coordination summary:


,panel_name,level,condition,group_name,n_donors,coordination_mean_pairwise_r,coordination_mean_pairwise_r_c9_complex,coordination_mean_pairwise_r_ulk_complex,coordination_mean_pairwise_r_between_complexes,coordination_pc1_variance_explained,coordination_flag
0,coordination_panel,broad,Control,Excitatory,4,-0.050722,-0.309054,0.287368,-0.224449,0.669432,tentative_low_n
1,coordination_panel,broad,Control,Glia,4,-0.009260,0.229676,-0.200532,0.070467,0.548959,tentative_low_n
2,coordination_panel,broad,Control,Inhibitory,4,-0.096663,0.659261,0.375126,-0.562374,0.667101,tentative_low_n
3,coordination_panel,broad,Control,Vascular,3,NaN,NaN,NaN,NaN,NaN,insufficient_donors


Subtype control-only coordination summary:


,panel_name,level,condition,group_name,n_donors,coordination_mean_pairwise_r,coordination_mean_pairwise_r_c9_complex,coordination_mean_pairwise_r_ulk_complex,coordination_mean_pairwise_r_between_complexes,coordination_pc1_variance_explained,coordination_flag
0,coordination_panel,subtype,Control,Ex.L2_L3.CUX2_RASGRF2,4,0.021329,-0.297385,0.530914,-0.254651,0.785192,tentative_low_n
1,coordination_panel,subtype,Control,Ex.L3_L5.CUX2_RORB,4,-0.089249,-0.312965,0.124922,-0.187287,0.525750,tentative_low_n
2,coordination_panel,subtype,Control,Ex.L4_L5.RORB_POU3F2,4,-0.116853,-0.294844,0.001724,-0.160307,0.605707,tentative_low_n
3,coordination_panel,subtype,Control,Ex.L4_L6.RORB_LRRK1,4,-0.027968,-0.310959,0.226806,-0.141218,0.450852,tentative_low_n
4,coordination_panel,subtype,Control,Ex.L5.PCP4_NXPH2,3,NaN,NaN,NaN,NaN,NaN,insufficient_donors
5,coordination_panel,subtype,Control,Ex.L5.VAT1L_THSD4,1,NaN,NaN,NaN,NaN,NaN,insufficient_donors
6,coordination_panel,subtype,Control,Ex.L5_L6.THEMIS_NR4A2,3,NaN,NaN,NaN,NaN,NaN,insufficient_donors
7,coordination_panel,subtype,Control,Ex.L5_L6.THEMIS_TMEM233,4,-0.062372,-0.266517,0.199417,-0.196069,0.700380,tentative_low_n
8,coordination_panel,subtype,Control,Ex.L6.TLE4_CCBE1,4,-0.086085,-0.269512,-0.065249,-0.063290,0.486752,tentative_low_n
9,coordination_panel,subtype,Control,Ex.L6.TLE4_MEGF11,4,-0.082957,-0.027462,0.332400,-0.370959,0.584610,tentative_low_n


### B3. Coordination Summary Metrics And B4. Selected Gene-Pair Visualisations

I summarise a small set of coordination metrics and I generate a selected scatter-plot panel for biologically central cross-complex gene pairs.


In [11]:
coord_broad_diff_summary = extract_coordination_difference_summary(
    coord_broad_condition_summary,
    comparison_pairs={
        'c9ALS_vs_Control': ('c9ALS', 'Control'),
        'c9ALS_vs_sALS': ('c9ALS', 'sALS'),
    },
)
coord_subtype_diff_summary = extract_coordination_difference_summary(
    coord_subtype_condition_summary,
    comparison_pairs={
        'c9ALS_vs_Control': ('c9ALS', 'Control'),
        'c9ALS_vs_sALS': ('c9ALS', 'sALS'),
    },
)

coord_broad_diff_summary.to_csv(TABLE_DIR / '15_coordination_panel_broad_difference_summary.csv', index=False)
coord_subtype_diff_summary.to_csv(TABLE_DIR / '15_coordination_panel_subtype_difference_summary.csv', index=False)

for group_name in SELECTED_SCATTER_GROUPS_BROAD:
    for gene_x, gene_y in SELECTED_SCATTER_PAIRS:
        plot_selected_gene_pair_scatter(
            coord_broad_baseline['logcpm'],
            broad_union_meta,
            group_col='group_name',
            group_name=group_name,
            gene_x=gene_x,
            gene_y=gene_y,
            level_name='broad',
            panel_name='coordination_panel',
        )

for group_name in SELECTED_SCATTER_GROUPS_SUBTYPE:
    for gene_x, gene_y in SELECTED_SCATTER_PAIRS:
        plot_selected_gene_pair_scatter(
            coord_subtype_baseline['logcpm'],
            subtype_union_meta,
            group_col='cell_subtype',
            group_name=group_name,
            gene_x=gene_x,
            gene_y=gene_y,
            level_name='subtype',
            panel_name='coordination_panel',
        )

print('Broad condition-specific coordination differences:')
display(coord_broad_diff_summary)
print('Subtype condition-specific coordination differences:')
display(coord_subtype_diff_summary.head(20))


Broad condition-specific coordination differences:


,group_name,comparison,coordination_delta_mean_pairwise_r,coordination_delta_pc1_variance,coordination_disruption_abs_delta
0,Excitatory,c9ALS_vs_Control,-0.062957,-0.049830,0.062957
1,Excitatory,c9ALS_vs_sALS,-0.032980,0.072139,0.032980
2,Glia,c9ALS_vs_Control,-0.044014,0.286538,0.044014
3,Glia,c9ALS_vs_sALS,0.028337,0.235495,0.028337
4,Inhibitory,c9ALS_vs_Control,0.113961,-0.042283,0.113961
5,Inhibitory,c9ALS_vs_sALS,0.114203,0.147319,0.114203
6,Vascular,c9ALS_vs_Control,NaN,NaN,NaN
7,Vascular,c9ALS_vs_sALS,NaN,NaN,NaN


Subtype condition-specific coordination differences:


,group_name,comparison,coordination_delta_mean_pairwise_r,coordination_delta_pc1_variance,coordination_disruption_abs_delta
0,Ex.L2_L3.CUX2_RASGRF2,c9ALS_vs_Control,-0.122348,-0.190439,0.122348
1,Ex.L2_L3.CUX2_RASGRF2,c9ALS_vs_sALS,0.029230,0.113771,0.029230
2,Ex.L3_L5.CUX2_RORB,c9ALS_vs_Control,0.103706,0.078363,0.103706
3,Ex.L3_L5.CUX2_RORB,c9ALS_vs_sALS,NaN,NaN,NaN
4,Ex.L4_L5.RORB_POU3F2,c9ALS_vs_Control,-0.004474,-0.108744,0.004474
5,Ex.L4_L5.RORB_POU3F2,c9ALS_vs_sALS,NaN,NaN,NaN
6,Ex.L4_L6.RORB_LRRK1,c9ALS_vs_Control,-0.072849,-0.022996,0.072849
7,Ex.L4_L6.RORB_LRRK1,c9ALS_vs_sALS,NaN,NaN,NaN
8,Ex.L5.PCP4_NXPH2,c9ALS_vs_Control,NaN,NaN,NaN
9,Ex.L5.PCP4_NXPH2,c9ALS_vs_sALS,NaN,NaN,NaN


## SECTION C. Optional WGCNA Integration And Follow-Up Ranking

If saved broad-cell-type WGCNA outputs are available in a clean tabular format, I summarise whether the coordination-panel genes fall into the same module. I then integrate the Claw-neighbourhood and coordination summaries into a simple follow-up ranking table.


In [12]:
wgcna_rows = []
wgcna_status_rows = []
for group_name in BROAD_CLASS_ORDER:
    subset, status = try_read_wgcna_membership_for_panel(group_name, present_coordination_genes)
    wgcna_status_rows.append({'group_name': group_name, 'status': status})
    if not subset.empty:
        module_counts = subset['module'].astype(str).value_counts().rename_axis('module').reset_index(name='n_panel_genes')
        dominant_module = module_counts.iloc[0]['module']
        subset['dominant_panel_module'] = dominant_module
        subset['n_genes_in_dominant_module'] = module_counts.iloc[0]['n_panel_genes']
        wgcna_rows.append(subset)

wgcna_status_df = pd.DataFrame(wgcna_status_rows)
wgcna_status_df.to_csv(TABLE_DIR / '15_optional_wgcna_status.csv', index=False)

if wgcna_rows:
    wgcna_membership_df = pd.concat(wgcna_rows, ignore_index=True)
    wgcna_membership_df.to_csv(TABLE_DIR / '15_optional_wgcna_coordination_panel_module_membership.csv', index=False)
    print('Optional WGCNA integration summary:')
    display(wgcna_membership_df)
else:
    print('No readable broad-cell-type WGCNA module-membership exports were found. I will skip WGCNA integration for this run.')
    display(wgcna_status_df)

broad_coord_focus = (
    coord_broad_diff_summary.loc[coord_broad_diff_summary['comparison'].eq('c9ALS_vs_Control')]
    .drop(columns=['comparison'])
    .copy()
)
subtype_coord_focus = (
    coord_subtype_diff_summary.loc[coord_subtype_diff_summary['comparison'].eq('c9ALS_vs_Control')]
    .drop(columns=['comparison'])
    .copy()
)

broad_followup_summary = build_followup_summary(
    level_name='broad',
    claw_baseline_summary=claw_broad_baseline['summary'],
    claw_perturbation_summary=claw_broad_perturbation_summary,
    coord_control_summary=coord_broad_control_summary,
    coord_diff_summary=broad_coord_focus,
    retention_summary=broad_retention_summary,
)
subtype_followup_summary = build_followup_summary(
    level_name='subtype',
    claw_baseline_summary=claw_subtype_baseline['summary'],
    claw_perturbation_summary=claw_subtype_perturbation_summary,
    coord_control_summary=coord_subtype_control_summary,
    coord_diff_summary=subtype_coord_focus,
    retention_summary=subtype_retention_summary,
)

print('Broad follow-up summary:')
display(broad_followup_summary)
print('Top subtype follow-up summary rows:')
display(subtype_followup_summary.head(20))


No readable broad-cell-type WGCNA module-membership exports were found. I will skip WGCNA integration for this run.


,group_name,status
0,Excitatory,Excitatory: no_plaintext_module_exports_found
1,Inhibitory,Inhibitory: no_saved_wgcna_directory
2,Glia,Glia: no_plaintext_module_exports_found
3,Vascular,Vascular: no_saved_wgcna_directory


Broad follow-up summary:


,group_name,claw_baseline_programme_score,n_control_donors,claw_perturbation_metric,claw_n_perturbed_genes,coordination_baseline_mean_pairwise_r,coordination_baseline_pc1_variance_explained,coordination_delta_mean_pairwise_r,coordination_delta_pc1_variance,coordination_disruption_abs_delta,min_retained_donors,claw_baseline_scaled,claw_perturbation_scaled,coordination_baseline_scaled,coordination_disruption_scaled,robustness_scaled,followup_score,level,followup_rank
0,Inhibitory,0.337912,4,0.179532,0,-0.096663,0.667101,0.113961,-0.042283,0.113961,4,1.000000,0.037432,0.000000,1.000000,1.0,2.537432,broad,1
1,Glia,-0.085769,4,0.222847,0,-0.009260,0.548959,-0.044014,0.286538,0.044014,4,0.344020,0.323908,0.904199,0.386221,1.0,2.458349,broad,2
2,Excitatory,0.055820,4,0.173873,0,-0.050722,0.669432,-0.062957,-0.049830,0.062957,4,0.563242,0.000000,0.475272,0.552442,1.0,2.090955,broad,3
3,Vascular,-0.307963,3,0.325069,0,NaN,NaN,NaN,NaN,NaN,3,0.000000,1.000000,1.000000,0.000000,0.0,2.000000,broad,4


Top subtype follow-up summary rows:


,group_name,claw_baseline_programme_score,n_control_donors,claw_perturbation_metric,claw_n_perturbed_genes,coordination_baseline_mean_pairwise_r,coordination_baseline_pc1_variance_explained,coordination_delta_mean_pairwise_r,coordination_delta_pc1_variance,coordination_disruption_abs_delta,min_retained_donors,claw_baseline_scaled,claw_perturbation_scaled,coordination_baseline_scaled,coordination_disruption_scaled,robustness_scaled,followup_score,level,followup_rank
0,Glia.Astro.GFAP.neg,0.078055,4,0.231153,1,0.135616,0.534406,-0.137412,0.055028,0.137412,4,0.757870,0.211914,1.000000,1.000000,1.0,3.469784,subtype,1
1,Ex.L2_L3.CUX2_RASGRF2,0.055938,4,0.198536,1,0.021329,0.785192,-0.122348,-0.190439,0.122348,4,0.703491,0.129946,0.557866,0.890378,1.0,2.781681,subtype,2
2,Glia.Astro.GFAP.pos,0.170542,4,0.544749,0,-0.122873,0.476847,0.034554,0.137524,0.034554,4,0.985271,1.000000,0.000000,0.251466,1.0,2.736737,subtype,3
3,Glia.OPC,0.176532,4,0.201082,2,-0.027779,0.546555,0.062965,0.122795,0.062965,4,1.000000,0.136342,0.367886,0.458219,1.0,2.462447,subtype,4
4,In.PV.PVALB_MYBPC1,0.077523,4,0.224686,0,-0.015279,0.497728,-0.076227,-0.018943,0.076227,4,0.756563,0.195661,0.416243,0.554733,1.0,2.423200,subtype,5
5,Glia.Micro,-0.018258,4,0.342123,0,-0.043230,0.471014,-0.065012,0.165351,0.065012,4,0.521061,0.490789,0.308111,0.473119,1.0,2.293080,subtype,6
6,Ex.L6.TLE4_CCBE1,0.115544,4,0.225502,0,-0.086085,0.486752,0.059527,0.025688,0.059527,4,0.850047,0.197713,0.142320,0.433202,1.0,2.123282,subtype,7
7,Ex.L3_L5.CUX2_RORB,-0.000079,4,0.284624,0,-0.089249,0.525750,0.103706,0.078363,0.103706,3,0.565760,0.346289,0.130078,0.754712,0.5,2.046838,subtype,8
8,Ex.L5.PCP4_NXPH2,0.132806,3,0.373906,0,NaN,NaN,NaN,NaN,NaN,2,0.892489,0.570662,0.475352,0.000000,0.0,1.938503,subtype,9
9,Ex.L4_L6.RORB_LRRK1,0.044166,4,0.146828,0,-0.027968,0.450852,-0.072849,-0.022996,0.072849,3,0.674547,0.000000,0.367155,0.530153,0.5,1.821856,subtype,10


## 6. Output Inventory And Draft Interpretation

I finish by recording what was saved and by generating a short draft summary that can be revised manually before any final biological claims are made.


In [14]:
table_files = sorted(str(path.relative_to(PROJECT_ROOT)) for path in TABLE_DIR.glob('*.csv'))[:400]
figure_files = sorted(str(path.relative_to(PROJECT_ROOT)) for path in FIG_DIR.glob('*.png'))[:400]
intermediate_files = sorted(str(path.relative_to(PROJECT_ROOT)) for path in INTERMEDIATE_DIR.glob('*.csv'))[:400]

max_len = max(len(table_files), len(figure_files), len(intermediate_files))

output_inventory = pd.DataFrame({
    'tables': table_files + [pd.NA] * (max_len - len(table_files)),
    'figures': figure_files + [pd.NA] * (max_len - len(figure_files)),
    'intermediate': intermediate_files + [pd.NA] * (max_len - len(intermediate_files)),
})

output_inventory.to_csv(TABLE_DIR / '15_output_inventory.csv', index=False)
display(output_inventory)


def summarise_top_hits(df, n=3):
    if df.empty:
        return []
    return df['group_name'].head(n).tolist()

robust_broad = broad_followup_summary.loc[
    broad_followup_summary['min_retained_donors'] >= MIN_DONORS_PER_CONDITION_FOR_SUBTYPE_DE,
    'group_name'
].tolist()
robust_subtypes = subtype_followup_summary.loc[
    subtype_followup_summary['min_retained_donors'] >= MIN_DONORS_PER_CONDITION_FOR_SUBTYPE_DE,
    'group_name'
].tolist()

summary_text = f"""
### Draft summary generated from the current notebook state

- **Most compelling broad classes right now:** {', '.join(summarise_top_hits(broad_followup_summary, n=3)) if not broad_followup_summary.empty else 'not yet available'}.
- **Most compelling refined subtypes right now:** {', '.join(summarise_top_hits(subtype_followup_summary, n=5)) if not subtype_followup_summary.empty else 'not yet available'}.
- **Broad classes with acceptable donor robustness under the current thresholds:** {', '.join(robust_broad) if robust_broad else 'none'}.
- **Refined subtypes with acceptable donor robustness under the current thresholds:** {', '.join(robust_subtypes[:10]) if robust_subtypes else 'none'}.
- **Main caution for interpretation:** condition-specific coordination changes and subtype DE should only be interpreted where donor counts were adequate; the retention, failure, and correlation-flag tables should be checked before making any final claims.
"""

display(Markdown(summary_text))


,tables,figures,intermediate
0,results/15_claw_neighbourhood_coordination/tab...,results/15_claw_neighbourhood_coordination/fig...,results/15_claw_neighbourhood_coordination/int...
1,results/15_claw_neighbourhood_coordination/tab...,results/15_claw_neighbourhood_coordination/fig...,results/15_claw_neighbourhood_coordination/int...
2,results/15_claw_neighbourhood_coordination/tab...,results/15_claw_neighbourhood_coordination/fig...,results/15_claw_neighbourhood_coordination/int...
3,results/15_claw_neighbourhood_coordination/tab...,results/15_claw_neighbourhood_coordination/fig...,results/15_claw_neighbourhood_coordination/int...
4,results/15_claw_neighbourhood_coordination/tab...,results/15_claw_neighbourhood_coordination/fig...,results/15_claw_neighbourhood_coordination/int...
5,results/15_claw_neighbourhood_coordination/tab...,results/15_claw_neighbourhood_coordination/fig...,results/15_claw_neighbourhood_coordination/int...
6,results/15_claw_neighbourhood_coordination/tab...,results/15_claw_neighbourhood_coordination/fig...,results/15_claw_neighbourhood_coordination/int...
7,results/15_claw_neighbourhood_coordination/tab...,results/15_claw_neighbourhood_coordination/fig...,results/15_claw_neighbourhood_coordination/int...
8,results/15_claw_neighbourhood_coordination/tab...,results/15_claw_neighbourhood_coordination/fig...,results/15_claw_neighbourhood_coordination/int...
9,results/15_claw_neighbourhood_coordination/tab...,results/15_claw_neighbourhood_coordination/fig...,results/15_claw_neighbourhood_coordination/int...



### Draft summary generated from the current notebook state

- **Most compelling broad classes right now:** Inhibitory, Glia, Excitatory.
- **Most compelling refined subtypes right now:** Glia.Astro.GFAP.neg, Ex.L2_L3.CUX2_RASGRF2, Glia.Astro.GFAP.pos, Glia.OPC, In.PV.PVALB_MYBPC1.
- **Broad classes with acceptable donor robustness under the current thresholds:** Inhibitory, Glia, Excitatory, Vascular.
- **Refined subtypes with acceptable donor robustness under the current thresholds:** Glia.Astro.GFAP.neg, Ex.L2_L3.CUX2_RASGRF2, Glia.Astro.GFAP.pos, Glia.OPC, In.PV.PVALB_MYBPC1, Glia.Micro, Ex.L6.TLE4_CCBE1, Ex.L3_L5.CUX2_RORB, Ex.L4_L6.RORB_LRRK1, In.Rosehip.LAMP5_PMEPA1.
- **Main caution for interpretation:** condition-specific coordination changes and subtype DE should only be interpreted where donor counts were adequate; the retention, failure, and correlation-flag tables should be checked before making any final claims.


### Manual Interpretation Placeholder

Please edit this section after reviewing the saved tables and figures.

- **Broad-level Claw-neighbourhood observations:**
  - [Edit after inspection]
- **Subtype-level Claw-neighbourhood observations:**
  - [Edit after inspection]
- **Broad-level coordination observations:**
  - [Edit after inspection]
- **Subtype-level coordination observations:**
  - [Edit after inspection]
- **Results that appear robust:**
  - [Edit after inspection]
- **Results that remain tentative because of donor retention or low correlation sample size:**
  - [Edit after inspection]
- **Manual decisions still pending:**
  - whether the Claw-neighbourhood perturbation thresholds should be adjusted for the final figure set
  - whether any subtype-level claims should be restricted further on robustness grounds
  - whether optional comparator panels should be added in a second pass
